In [1]:
import os

In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
# from langchain.chains import LLMChain
# from langchain.chains import SequentialChain
import os
import json
import pandas as pd
import traceback
from dotenv import load_dotenv
from pypdf import PdfReader
from pydantic import BaseModel, Field
from pypdf import PdfReader

In [3]:
load_dotenv()

True

In [4]:
key=os.getenv("genai_api_key")

In [ ]:
key

In [6]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    google_api_key=key,
    temperature=0.5
)

In [7]:
print(llm)

metadata={'lc_versions': {'langchain-core': '1.6.2', 'langchain': '1.4.0', 'langchain-google-genai': '4.4.0'}} profile={'name': 'Gemini 3.5 Flash', 'release_date': '2026-05-19', 'last_updated': '2026-05-19', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True, 'reasoning_effort_levels': ['minimal', 'low', 'medium', 'high'], 'reasoning_effort_default': 'medium'} google_api_key=SecretStr('**********') model='gemini-3.5-flash' temperature=0.5 client=<google.genai.client.Client object at 0x0000014C697B27D0> default_metadata=() model_kwargs={}


In [8]:
RESPONSE_JSON = {
    "1": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    },
    "2": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    },
    "3": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    },
}


In [9]:
print(RESPONSE_JSON)

{'1': {'mcq': 'multiple choice question', 'options': {'a': 'choice here', 'b': 'choice here', 'c': 'choice here', 'd': 'choice here'}, 'correct': 'correct answer'}, '2': {'mcq': 'multiple choice question', 'options': {'a': 'choice here', 'b': 'choice here', 'c': 'choice here', 'd': 'choice here'}, 'correct': 'correct answer'}, '3': {'mcq': 'multiple choice question', 'options': {'a': 'choice here', 'b': 'choice here', 'c': 'choice here', 'd': 'choice here'}, 'correct': 'correct answer'}}


In [10]:

TEMPLATE="""
Text:{text}
You are an expert MCQ maker. Given the above text, it is your job to \
create a quiz  of {number} multiple choice questions for {subject} students in {tone} tone. 
Make sure the questions are not repeated and check all the questions to be conforming the text as well.
Make sure to format your response like  RESPONSE_JSON below  and use it as a guide. \
Ensure to make {number} MCQs
### RESPONSE_JSON
{Response_json}

"""

In [11]:
from langchain_core.prompts import PromptTemplate


In [12]:
quiz_generation_prompt = PromptTemplate(
    input_variables=["text", "number", "subject", "tone", "Response_json"],
    template=TEMPLATE
)

In [13]:
print(quiz_generation_prompt)

input_variables=['Response_json', 'number', 'subject', 'text', 'tone'] input_types={} partial_variables={} template='\nText:{text}\nYou are an expert MCQ maker. Given the above text, it is your job to create a quiz  of {number} multiple choice questions for {subject} students in {tone} tone. \nMake sure the questions are not repeated and check all the questions to be conforming the text as well.\nMake sure to format your response like  RESPONSE_JSON below  and use it as a guide. Ensure to make {number} MCQs\n### RESPONSE_JSON\n{Response_json}\n\n'


In [14]:
quiz_chain = quiz_generation_prompt | llm

In [15]:
TEMPLATE2="""
You are an expert english grammarian and writer. Given a Multiple Choice Quiz for {subject} students.\
You need to evaluate the complexity of the question and give a complete analysis of the quiz. Only use at max 50 words for complexity analysis. 
if the quiz is not at per with the cognitive and analytical abilities of the students,\
update the quiz questions which needs to be changed and change the tone such that it perfectly fits the student abilities
Quiz_MCQs:
{quiz}

Check from an expert English Writer of the above quiz:
"""

In [16]:
quiz_evaluation_prompt=PromptTemplate(input_variables=["subject", "quiz"], template=TEMPLATE2)

In [17]:
review_chain= quiz_evaluation_prompt | llm

In [18]:
from langchain_core.runnables import RunnableLambda

In [19]:
def generate_quiz(inputs):
    response = quiz_chain.invoke(inputs)
    

    return {
        "subject": inputs["subject"],
        "quiz": response.content
    }


def review_quiz(inputs):
    response = review_chain.invoke({
        "subject": inputs["subject"],
        "quiz": inputs["quiz"]
    })

    return {
        "quiz": inputs["quiz"],
        "review": response.content
    }


generate_evaluate_chain = (
    RunnableLambda(generate_quiz)
    | RunnableLambda(review_quiz)
)

In [20]:
file_path = r"C:\Users\namde\OneDrive\Desktop\AishuNamdev2006AishuNamdev2006--MCQ-Generator-using-gemini-Langchain-Streamlit\data.txt"

In [21]:
print(file_path)

C:\Users\namde\OneDrive\Desktop\AishuNamdev2006AishuNamdev2006--MCQ-Generator-using-gemini-Langchain-Streamlit\data.txt


In [22]:
with open(file_path, "r", encoding="utf-8") as file:
    TEXT = file.read()

print(TEXT)

What is Generative AI?
Generative artificial intelligence (generative AI) is a type of AI that can create new content and ideas, including conversations, stories, images, videos, and music. It can learn human language, programming languages, art, chemistry, biology, or any complex subject matter. It reuses what it knows to solve new problems. For example, it can learn English vocabulary and create a poem from the words it processes. Your organization can use generative AI for various purposes, like chatbots, media creation, product development, and design.

What is the difference between AI and Generative AI?
Artificial intelligence is the broader concept of making machines more human-like. It includes everything from smart assistants like Alexa, chatbots, and image generators to robotic vacuum cleaners and self-driving cars. Generative AI is a subset that generates new content meaningfully and intelligently.

When was generative AI created?
Generative AI emerged in the late 2010s with

In [23]:
json.dumps(RESPONSE_JSON)

'{"1": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}, "2": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}, "3": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}}'

In [24]:
NUMBER=5 
SUBJECT="genrative ai"
TONE="simple"

In [37]:
mcq_result = generate_evaluate_chain.invoke({
    "text": TEXT,
    "number": NUMBER,
    "subject": SUBJECT,
    "tone": TONE,
    "Response_json": json.dumps(RESPONSE_JSON)
})



In [38]:
print(type(mcq_result))
print(mcq_result)

<class 'dict'>
{'quiz': [{'type': 'text', 'text': '```json\n{\n  "1": {\n    "mcq": "What is the main difference between Artificial Intelligence (AI) and Generative AI?",\n    "options": {\n      "a": "AI is a subset of Generative AI that only focuses on generating text.",\n      "b": "AI is the broader concept of making machines more human-like, while Generative AI is a subset that generates new content.",\n      "c": "Generative AI is used only for robotic vacuum cleaners, while AI is used for chatbots.",\n      "d": "There is no difference; both terms mean the exact same thing."\n    },\n    "correct": "b"\n  },\n  "2": {\n    "mcq": "How do Generative Adversarial Networks (GANs) work?",\n    "options": {\n      "a": "By training two neural networks—a generator and a discriminator—in a competitive manner.",\n      "b": "By using a single encoder network to compress data into a latent space.",\n      "c": "By progressively adding noise to an image and then reversing the process.",\n 

In [39]:
mcq_result

{'quiz': [{'type': 'text',
   'text': '```json\n{\n  "1": {\n    "mcq": "What is the main difference between Artificial Intelligence (AI) and Generative AI?",\n    "options": {\n      "a": "AI is a subset of Generative AI that only focuses on generating text.",\n      "b": "AI is the broader concept of making machines more human-like, while Generative AI is a subset that generates new content.",\n      "c": "Generative AI is used only for robotic vacuum cleaners, while AI is used for chatbots.",\n      "d": "There is no difference; both terms mean the exact same thing."\n    },\n    "correct": "b"\n  },\n  "2": {\n    "mcq": "How do Generative Adversarial Networks (GANs) work?",\n    "options": {\n      "a": "By training two neural networks—a generator and a discriminator—in a competitive manner.",\n      "b": "By using a single encoder network to compress data into a latent space.",\n      "c": "By progressively adding noise to an image and then reversing the process.",\n      "d": "B

In [40]:
result = llm.invoke("What is Generative AI?")

print(result.content)

[{'type': 'text', 'text': '**Generative Artificial Intelligence (Generative AI or GenAI)** is a branch of artificial intelligence capable of generating new content—such as text, images, music, audio, video, and computer code—in response to user prompts. \n\nUnlike traditional AI, which is designed to analyze data, find patterns, or make decisions based on existing information, Generative AI **creates something entirely new** that resembles human-made content.\n\n---\n\n### 1. How is it different from Traditional AI?\nTo understand Generative AI, it helps to compare it to traditional (or "analytical") AI:\n\n*   **Traditional AI (Analytical):** It looks at data and tells you what it is. \n    *   *Example:* You show the AI thousands of photos of cats, and it learns to identify if a new photo contains a cat. (Classification/Prediction).\n*   **Generative AI:** It looks at data and creates something new based on what it learned.\n    *   *Example:* You type, "Draw a cat wearing a spacesui

In [41]:
usage = result.usage_metadata

print(f"Total Tokens: {usage.get('total_tokens', 0)}")
print(f"Prompt Tokens: {usage.get('input_tokens', 0)}")
print(f"Completion Tokens: {usage.get('output_tokens', 0)}")

Total Tokens: 2175
Prompt Tokens: 7
Completion Tokens: 2168


In [42]:
quiz = result.content

In [43]:
print(quiz[0])

{'type': 'text', 'text': '**Generative Artificial Intelligence (Generative AI or GenAI)** is a branch of artificial intelligence capable of generating new content—such as text, images, music, audio, video, and computer code—in response to user prompts. \n\nUnlike traditional AI, which is designed to analyze data, find patterns, or make decisions based on existing information, Generative AI **creates something entirely new** that resembles human-made content.\n\n---\n\n### 1. How is it different from Traditional AI?\nTo understand Generative AI, it helps to compare it to traditional (or "analytical") AI:\n\n*   **Traditional AI (Analytical):** It looks at data and tells you what it is. \n    *   *Example:* You show the AI thousands of photos of cats, and it learns to identify if a new photo contains a cat. (Classification/Prediction).\n*   **Generative AI:** It looks at data and creates something new based on what it learned.\n    *   *Example:* You type, "Draw a cat wearing a spacesuit

In [44]:
import json

quiz_text =mcq_result["quiz"][0]["text"]

# Remove markdown code fences
quiz_text = quiz_text.replace("```json", "").replace("```", "").strip()

# Convert JSON string to Python dictionary
quiz = json.loads(quiz_text)

print(quiz)

{'1': {'mcq': 'What is the main difference between Artificial Intelligence (AI) and Generative AI?', 'options': {'a': 'AI is a subset of Generative AI that only focuses on generating text.', 'b': 'AI is the broader concept of making machines more human-like, while Generative AI is a subset that generates new content.', 'c': 'Generative AI is used only for robotic vacuum cleaners, while AI is used for chatbots.', 'd': 'There is no difference; both terms mean the exact same thing.'}, 'correct': 'b'}, '2': {'mcq': 'How do Generative Adversarial Networks (GANs) work?', 'options': {'a': 'By training two neural networks—a generator and a discriminator—in a competitive manner.', 'b': 'By using a single encoder network to compress data into a latent space.', 'c': 'By progressively adding noise to an image and then reversing the process.', 'd': 'By predicting the next word in a sequence using probability distribution.'}, 'correct': 'a'}, '3': {'mcq': 'Which of the following is recommended as a 

In [45]:
# 7. Convert Quiz into Table Data

quiz_table_data = []

for question in quiz.values():

    options_text = " | ".join(
        f"{option}: {option_value}"
        for option, option_value in question["options"].items()
    )

    quiz_table_data.append({
        "MCQ": question["mcq"],
        "Choices": options_text,
        "Correct": question["correct"]
    })

print(quiz_table_data)

[{'MCQ': 'What is the main difference between Artificial Intelligence (AI) and Generative AI?', 'Choices': 'a: AI is a subset of Generative AI that only focuses on generating text. | b: AI is the broader concept of making machines more human-like, while Generative AI is a subset that generates new content. | c: Generative AI is used only for robotic vacuum cleaners, while AI is used for chatbots. | d: There is no difference; both terms mean the exact same thing.', 'Correct': 'b'}, {'MCQ': 'How do Generative Adversarial Networks (GANs) work?', 'Choices': 'a: By training two neural networks—a generator and a discriminator—in a competitive manner. | b: By using a single encoder network to compress data into a latent space. | c: By progressively adding noise to an image and then reversing the process. | d: By predicting the next word in a sequence using probability distribution.', 'Correct': 'a'}, {'MCQ': 'Which of the following is recommended as a best practice for organizations adopting 

In [46]:
quiz_table_data

[{'MCQ': 'What is the main difference between Artificial Intelligence (AI) and Generative AI?',
  'Choices': 'a: AI is a subset of Generative AI that only focuses on generating text. | b: AI is the broader concept of making machines more human-like, while Generative AI is a subset that generates new content. | c: Generative AI is used only for robotic vacuum cleaners, while AI is used for chatbots. | d: There is no difference; both terms mean the exact same thing.',
  'Correct': 'b'},
 {'MCQ': 'How do Generative Adversarial Networks (GANs) work?',
  'Choices': 'a: By training two neural networks—a generator and a discriminator—in a competitive manner. | b: By using a single encoder network to compress data into a latent space. | c: By progressively adding noise to an image and then reversing the process. | d: By predicting the next word in a sequence using probability distribution.',
  'Correct': 'a'},
 {'MCQ': 'Which of the following is recommended as a best practice for organizations

In [47]:
# 8. Create Pandas DataFrame
quiz_df = pd.DataFrame(quiz_table_data)


In [48]:
# 9. Display Quiz
print(quiz_df.to_string(index=False))


# 10. Save CSV
quiz_df.to_csv(
    "machinelearning.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nQuiz successfully saved to machinelearning.csv")


                                                                                                              MCQ                                                                                                                                                                                                                                                                                                                                                                Choices Correct
                              What is the main difference between Artificial Intelligence (AI) and Generative AI? a: AI is a subset of Generative AI that only focuses on generating text. | b: AI is the broader concept of making machines more human-like, while Generative AI is a subset that generates new content. | c: Generative AI is used only for robotic vacuum cleaners, while AI is used for chatbots. | d: There is no difference; both terms mean the exact same thing.       b
                                      

In [49]:
from datetime import datetime
datetime.now().strftime('%m_%d_%Y_%H_%M_%S')

'09_12_2026_17_56_53'